In [ ]:
# CasADi

tic = time.time()


alpha = 99/(fov_rng**2)#this is the distance decay variable (higher = more decay) 
# ^ (Currently, this does the math to make the k value <=0.01 outside of fov range)
c = 100 #this is the steepness of visibility transition (higher = sharper transition)

#MAX_K_VALUE = 0.9999

# New version, uses call to Camera.py to find FOV center:
def calculate_k_value(xi, yi, ti, camera_objects):
    total_cost = 0
    for cam in camera_objects:
        cam_x, cam_y = cam.cam_position()
        phi = cam.get_ctr_theta_t(ti)  # Use Camera.py method

        theta = cam.fov_ang
        dx = xi - cam_x
        dy = yi - cam_y
        dist2 = dx**2 + dy**2
        # dist = ca.sqrt(dist2 + 1e-6)

        dist_min_sq = ca.fmax(dist2, 1e-4) #try putting this back to 1e-4 if the optimization works
        dist_safe = ca.sqrt(dist_min_sq)

        cos_alpha = (dx * ca.cos(phi) + dy * ca.sin(phi)) / dist_safe
        visibility = 1 / (1 + ca.exp(-c * (cos_alpha - ca.cos(theta / 2))))
        observability = 1 / (1 + alpha * dist_min_sq)
        total_cost += visibility * observability
    # return ca.fmin(total_cost,MAX_K_VALUE) # This prevents ln(0) issues which break the solver
    return total_cost

opti = ca.Opti()


# MULTIPLE SHOOTING TIME BABYYYYY

# --- 1. Use RRT Path Directly for Parameters ---
# Extract arrays from the RRT path list [[x,y,t], ...]
path_arr = np.array(path)
rrt_x = path_arr[:, 0]
rrt_y = path_arr[:, 1]
rrt_t = path_arr[:, 2]

# Set N_shooting based on the number of segments in the RRT path
N_shooting = len(path) - 1 
M_steps = 100  # Integration steps per interval (keep this high for collision safety)
N = N_shooting * M_steps  # Total number of time steps (for plotting later)

# --- Decision Variables ---
# State: [x, y, t] at the boundaries of shooting intervals
X = opti.variable(3, N_shooting + 1)
pos_x = X[0, :]
pos_y = X[1, :]
current_t = X[2, :]

# Controls: [v_x, v_y] for each shooting interval
U = opti.variable(2, N_shooting)

# Interval Duration: Time duration of each shooting interval
T_interval = opti.variable(N_shooting)

# --- Initial/Final Constraints ---
opti.subject_to(pos_x[0] == x0_1[0])
opti.subject_to(pos_y[0] == x0_1[1])
opti.subject_to(current_t[0] == x0_1[2])

opti.subject_to(pos_x[-1] == xf[0])
opti.subject_to(pos_y[-1] == xf[1])

# --- Generate Camera Objects ---
camera_objects = []
for i in range(cam_dict['n']):
    cam_obj = Camera(i, cam_dict, cam_dict['x'][i], cam_dict['y'][i])
    camera_objects.append(cam_obj)

# --- Multiple Shooting Loop ---
log_sum = 0
epsilon = 1e-6
vmax = vehicle['v']
dt_min = 1e-4 

vehicle_radius = 0 # vehicle['radius'] # From vehicle setup             # PUT THIS BACK!!!
buffer_dist = 0    # Extra buffer; Adjust this in both the rrt step and casadi step (make sure they match)
safety_margin = vehicle_radius + buffer_dist

gamma = 500 # smoothing factor for log-sum-exp approximation

for k in range(N_shooting): # Iterating over each rrt point
    # 1. Get state at start of interval k
    st = X[:, k]  
    # 2. Get control for interval k
    v_k = U[:, k] 
    
    # 3. Time integration step size for this interval
    dt = T_interval[k] / M_steps
    opti.subject_to(dt >= dt_min)
    
    # 4. Integrate forward M steps (Euler Integration)
    for j in range(M_steps):
        # Cost at current position and time
        k_val = calculate_k_value(st[0], st[1], st[2], camera_objects)
        log_sum += ca.log(ca.fmax(1 - k_val, epsilon))
        
        # Map Bounds
        opti.subject_to(opti.bounded(map_size[0], st[0], map_size[1]))
        opti.subject_to(opti.bounded(map_size[2], st[1], map_size[3]))
        
        # Static Obstacles
        p_k = st[0:2] 
        for obs_data in obstacle_constraints:
            A = obs_data['A']
            b = obs_data['b']
            h_i_vector = ca.mtimes(A, p_k) - b
            
            # Number of edges in this specific polygon
            num_edges = A.shape[0]

            # Log-Sum-Exp Constraint for Obstacle avoidance
            max_h = h_i_vector[0]
            for idx in range(1, num_edges):
                max_h = ca.fmax(max_h, h_i_vector[idx])
            
            lse_term = max_h + (1.0/gamma) * ca.log(ca.sum(ca.exp(gamma * (h_i_vector - max_h))))

            # --- THE FIX: SUBTRACT THE SMOOTHING ERROR ---
            # We subtract log(num_edges)/gamma to ensure the constraint is strict.
            # This closes the "numerical gap" at the corners.
            lse_debiased = lse_term - (ca.log(num_edges) / gamma)

            # opti.subject_to(lse_term >= safety_margin)
            opti.subject_to(lse_debiased >= safety_margin)

        # Dynamics Update (Euler)
        # x_next = x + v_x * dt
        st[0] = st[0] + v_k[0] * dt # x update
        st[1] = st[1] + v_k[1] * dt # y update
        st[2] = st[2] + dt          # t update

    # 5. Continuity Constraint (between shooting intervals)
    opti.subject_to(X[:, k+1] == st)
    
    # 6. Velocity Constraint
    opti.subject_to(v_k[0]**2 + v_k[1]**2 <= vmax**2)

# --- Total Time Constraint ---
T_total = ca.sum(T_interval) # This probably needs to be ca.sum1(ca.sum2())
opti.subject_to(T_total <= end_time)

# --- Objective ---
time_penalty = 0 #0.01 * T_total # Switch this back eventually
J_total_cost = log_sum - time_penalty 
opti.minimize(-J_total_cost)

# --- 2. Set Initial Guesses Directly from RRT Data ---

# State Guess: The RRT nodes exactly align with our X variables
X_init = ca.vertcat(rrt_x, rrt_y, rrt_t)
opti.set_initial(pos_x, rrt_x)
opti.set_initial(pos_y, rrt_y)
opti.set_initial(current_t, rrt_t)

# Calculate differentials for Control/Time guesses
dx_guess = np.diff(rrt_x)
dy_guess = np.diff(rrt_y)
dt_guess = np.diff(rrt_t)

# Prevent division by zero if RRT has duplicate time stamps (rare but possible)
dt_guess = np.maximum(dt_guess, 1e-4)

vx_guess = dx_guess / dt_guess
vy_guess = dy_guess / dt_guess

# Control Guess

U_init = [vx_guess, vy_guess]

opti.set_initial(U[0, :], vx_guess)
opti.set_initial(U[1, :], vy_guess)

# Time Interval Guess
T_interval_init = dt_guess
opti.set_initial(T_interval, dt_guess)


def interpolate_rrt_path_preserve_speed(rrt_path, num_points):
    """
    Interpolates the original STP-RRT path using time-based interpolation,
    preserving the original speed profile.

    Parameters:
    - rrt_path: list of [x, y, t] points from STP-RRT
    - num_points: number of interpolated points to generate

    Returns:
    - x_interp: list of interpolated x positions
    - y_interp: list of interpolated y positions
    - t_interp: list of interpolated time values
    """
    rrt_array = np.array(rrt_path)
    x = rrt_array[:, 0]
    y = rrt_array[:, 1]
    t = rrt_array[:, 2]

    # Create interpolation functions for x(t) and y(t)
    fx = interp1d(t, x, kind='linear')
    fy = interp1d(t, y, kind='linear')

    # Generate evenly spaced time values
    t_interp = np.linspace(t[0], t[-1], num_points)

    # Interpolate positions
    x_interp = fx(t_interp)
    y_interp = fy(t_interp)

    return x_interp.tolist(), y_interp.tolist(), t_interp.tolist()



# # Interpolating RRT initial guess
x_interp, y_interp, t_interp = interpolate_rrt_path_preserve_speed(path, N+1)


# ... (previous code for setting up opti, X, U, T_interval, and the loop) ...

# --- RE-IMPLEMENTED EVALUATOR ---

# 1. Define a CasADi function that takes the distinct matrices as inputs
# Inputs: X (3 x N+1), U (2 x N), T_interval (N)
# Outputs: log_sum, time_penalty
cost_evaluator = ca.Function('cost_evaluator', [X, U, T_interval], [log_sum, time_penalty])

# 2. Prepare Numerical Initial Guesses
# X_init needs to be shape (3, N_shooting+1)
X_num_init = np.vstack([rrt_x, rrt_y, rrt_t]) 

# U_init needs to be shape (2, N_shooting)
U_num_init = np.vstack([vx_guess, vy_guess])

# T_init needs to be shape (N_shooting,)
T_num_init = dt_guess

# 3. Calculate Initial Probability
init_log_sum_dm, init_penalty_dm = cost_evaluator(X_num_init, U_num_init, T_num_init)
init_log_sum_val = float(init_log_sum_dm)

initial_P_detection = 1 - np.exp(init_log_sum_val)

# --- Solve Optimization ---
p_opts = {"expand": True}
s_opts = {
    "max_iter": 3000,
    "tol": 1e-7, # 1e-5,
    "acceptable_tol": 1e-5, # 1e-4,
    "constr_viol_tol": 1e-7, # Ensure constraint violation is minimal
    "print_level": 5
}
opti.solver("ipopt", p_opts, s_opts)

try:
    sol = opti.solve()
    
    # Extract solution values
    X_num_opt = sol.value(X)
    U_num_opt = sol.value(U)
    T_num_opt = sol.value(T_interval)
    
    # 4. Calculate Final Probability using the same evaluator
    final_log_sum_dm, final_penalty_dm = cost_evaluator(X_num_opt, U_num_opt, T_num_opt)
    final_log_sum_val = float(final_log_sum_dm)
    
    final_P_detection = 1 - np.exp(final_log_sum_val)

    # Prepare plotting data (reconstruct dense path)
    x_nodes = X_num_opt[0, :]
    y_nodes = X_num_opt[1, :]
    t_nodes = X_num_opt[2, :]
    
    x_opt = []
    y_opt = []
    t_opt_num = []
    
    # Linearly interpolate between shooting nodes for visualization
    for k in range(N_shooting):
        x_seg = np.linspace(x_nodes[k], x_nodes[k+1], M_steps, endpoint=False)
        y_seg = np.linspace(y_nodes[k], y_nodes[k+1], M_steps, endpoint=False)
        t_seg = np.linspace(t_nodes[k], t_nodes[k+1], M_steps, endpoint=False)
        
        x_opt.extend(x_seg)
        y_opt.extend(y_seg)
        t_opt_num.extend(t_seg)
    
    # Append final point
    x_opt.append(x_nodes[-1])
    y_opt.append(y_nodes[-1])
    t_opt_num.append(t_nodes[-1])

except RuntimeError as e:
    print("Solver failed:", e)
    # Fallback to debug values if solve fails
    x_opt = opti.debug.value(pos_x)
    y_opt = opti.debug.value(pos_y)
    t_opt_num = opti.debug.value(current_t)
    final_P_detection = 1.0 # Sentinel value

toc = time.time() - tic
print(f"Initial Probability of Detection: {initial_P_detection:.4f}")
print(f"Final Probability of Detection: {final_P_detection:.4f}")
print(f"Time to calculate: {toc:.4f} s")
